In [1]:
!pip uninstall -y massi
!pip cache purge

Found existing installation: massi 0.1.0
Uninstalling massi-0.1.0:
  Successfully uninstalled massi-0.1.0
Files removed: 6 (1.2 MB)


In [2]:
!cd .. && pip install -v . --group=dev --group=observability --group=sandbox

Using pip 25.3 from /opt/conda/lib/python3.13/site-packages/pip (python 3.13)
Processing /home/jovyan/work
  Running command installing build dependencies
  Using pip 25.3 from /opt/conda/lib/python3.13/site-packages/pip (python 3.13)
    Obtaining dependency information for setuptools>=40.8.0 from https://files.pythonhosted.org/packages/9d/76/f789f7a86709c6b087c5a2f52f911838cad707cc613162401badc665acfe/setuptools-82.0.1-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.0 MB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.7 MB/s  0:00:00
  Installing build dependencies ... done
  Running command Getting requirements to build wheel
  running egg_info
  writing massi.egg-info/PKG-INFO
  writing dependency_links to massi.egg-info/dependency_links.txt
  writing requirements to massi.egg-info/requires.txt
  writing top-level names to massi.egg-info/top_level.txt
  reading manifest file 'massi.egg-info/SOURCES.txt'
  writing manifest file

In [3]:
# Setup observability
USE_OBSERVABILITY = True

if USE_OBSERVABILITY:
    import os
    from lmnr import Laminar
    
    Laminar.initialize(
        project_api_key=os.environ["LMNR_PROJECT_API_KEY"],
        base_url=f"http://{os.environ["HOST_MACHINE_URL_CORE"]}",
        http_port=int(os.environ["LMNR_PROJECT_HTTP_PORT"]),
        grpc_port=int(os.environ["LMNR_PROJECT_GRPC_PORT"]),
    )
    print("Observability is ready")

Observability is ready


In [4]:
import lmnr
lmnr.version

<module 'lmnr.version' from '/opt/conda/lib/python3.13/site-packages/lmnr/version.py'>

In [5]:
from massi.agents.massi import Config

In [6]:
from massi.agents.massi import create_ollama_llm

base_config = Config(**{"model_url": "http://host.docker.internal:11434", "model_name": "granite4:3b"})
base_llm_client=create_ollama_llm(base_config)

In [7]:
from massi.agents.massi import create_openai_llm
openai_config = Config(**{"api_key": os.environ["OPENAI_API_TOKEN"], "model_name": "gpt-5.4-mini"})
openai_llm_client = create_openai_llm(openai_config)

In [8]:
from massi.agents.team.operators.author_proxy import create_openai_embeddings
open_ai_embeddings_config = Config(model_name="text-embedding-3-small", api_key=os.environ["OPENAI_API_TOKEN"])
open_ai_embeddings = create_openai_embeddings(open_ai_embeddings_config)

In [9]:
USE_OPEN_AI = True

print(">>> Using OpenAI") if USE_OPEN_AI else print(">>> Using Ollama")
main_llm = openai_llm_client if USE_OPEN_AI else base_llm_client
main_embeddings = open_ai_embeddings if USE_OPEN_AI else None

>>> Using OpenAI


# Starting the `EML`-x-`Author` massi

In [10]:
assert main_embeddings, "Can not proceed witht the Author-Focused execution without an embedding model"

In [11]:
from pathlib import Path
import massi.agents.team.operators.author_proxy as maupy

rag_author_proxy = maupy.RagAuthorProxyQueryClient.build_client(
    [Path('papers/budburst-paper.pdf')],
    {"embedder": main_embeddings},
    {"generator": main_llm},
)

In [12]:
import massi.agents.workflow as mw 

In [13]:
import massi.agents.team.architect as march
import massi.agents.team.planner as mplan

architect_map = {
    "darwin_core": march.DarwinCoreArchitect(main_llm),
    "eml": march.EMLArchitect(main_llm),
}
planner_map = {
    "darwin_core": mplan.DarwinCorePlanner(main_llm),
    "eml": mplan.EMLPlanner(main_llm),
}

In [11]:
import massi.agents.team.managers as mmaners
import massi.agents.team.operators as mopers

derivation_map = {
    'data-focus': mopers.DataGapDervationOperator(main_llm, recursion_limit=25, turn_limit=5),
    'author-focus': mopers.Au,
}

derivation_manager_map = {
    'data-focus': mmaners.DataGapDerivationManager(main_llm),
    'author-focus': ...,
}

AttributeError: module 'massi.agents.team.operators' has no attribute 'Au'

In [ ]:
dependency_base: mw.Dependencies = dict(
    execution_mode='author-focus',

    architect_map=architect_map,
    planner_map=planner_map,
    gap_derivation_manager_map=derivation_manager_map,
    derivation_map=derivation_map,
)

In [ ]:
resolution_dependencies = mw.Dependencies(**dependency_base)

In [ ]:
goal_workflow = mw._build_goal_resolution_graph(resolution_dependencies)

In [ ]:
goal_workflow.graph

In [ ]:
core_dependencies = mw.Dependencies(**dependency_base, metadata_resolution_subgraph=goal_workflow)

In [ ]:
core_workflow = mw.build_graph(core_dependencies)

In [ ]:
core_workflow.graph

## Inference

In [ ]:
import massi.agents.tools as mt
import massi.agents.team._dto as mdto

In [ ]:
dwc_data_file_metadata_goal: mw.MetadataGoal = {
    "level": mdto.MetadataLevel.ITEM,
    "scopes": [mdto.MetadataScope.DESCRIPTIVE, mdto.MetadataScope.TECHNICAL, mdto.MetadataScope.USE],
    "target_filename": "meta.xml",
    "schema_source": "https://dwc.tdwg.org/text/tdwg_dwc_text.xsd",
    "description": (
        "This metadata file describes the archive and data file structure."
        "A descriptor metafile describes how the files in your archive are organized."
        "It describes the files in the archive and maps each data column to a corresponding standard Darwin Core or Extension term."
        "The metafile is a relatively simple XML file format."
    ),
    "metadata_goal_key": "darwin_core",
}

In [ ]:
print(mt.get_xmlschema_validator(dwc_data_file_metadata_goal["schema_source"]).to_string())

In [ ]:
dwc_eml_metadata_goal: mw.MetadataGoal = {
    "level": mdto.MetadataLevel.COLLECTION,
    "scopes": [
        mdto.MetadataScope.DESCRIPTIVE,
        mdto.MetadataScope.ADMINISTRATIVE,
        mdto.MetadataScope.PRESERVATION,
        mdto.MetadataScope.USE,
    ],
    "target_filename": "eml.xml",
    "schema_source": "https://eml.ecoinformatics.org/eml-2.2.0/eml.xsd",
    "description": (
        "This metadata file contains the dataset's aggregate metadata."
        "This is a resource metadata document that provides information about the dataset itself such as a description (abstract) of the dataset, the agents responsible for authorship, publication and documentation, bibliographic and citation information, collection methods and much more."
        "It is based on the Ecological Metadata Language."
    ),
    "metadata_goal_key": "eml",
}

In [ ]:
print(mt.get_xmlschema_validator(dwc_eml_metadata_goal["schema_source"]).to_string())

## Mask available metadata

In [ ]:
!rm -rf dwc_data/masked-doi-10.34894-72gq4c/meta.xml
!rm -rf dwc_data/masked-doi-10.34894-72gq4c/eml.xml

## Run the inference

In [ ]:
import random

In [ ]:
thread_config = {"configurable": {"thread_id": str(random.randint(1, 100))}}

base_state: mw.MassiCoreState = {
    "settings": {
        "runtime_config": {
            "working_directory": "./dwc_data/masked-doi-10.34894-72gq4c", 
            "output_directory": "./dwc_data/masked-doi-10.34894-72gq4c"
        },
        "execution_mode": mw.MassiExecutionMode.DATA_INQUIRY
    },
    "goals": [dwc_eml_metadata_goal]
}

In [ ]:
outcome = core_workflow.graph.invoke(base_state, thread_config)

In [ ]:
from pprint import pprint

In [ ]:
print(outcome["resolutions"][0]["architect_xml_blueprint"]["content"])

In [ ]:
pprint(outcome["resolutions"][0]["gap_derivation_plan"])

In [ ]:
print(outcome["resolutions"][0]["working_copy_xml_blueprint"]["content"])